In [1]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv
# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip
# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"
print("Virtual environment ready with vLLM installed!")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127

In [2]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",                 # sm75: no bf16, no FlashAttention
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()


launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 4627, logging to /content/server.log


In [5]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 87s: http://localhost:8000/v1/models -> 200


In [6]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)
print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [7]:
import asyncio, time, httpx

async def send_one(client, base_url, model, prompt, max_tokens=128):
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "temperature": 0.0}
    t0 = time.perf_counter()
    try:
        r = await client.post(f"{base_url}/chat/completions", json=payload)
        r.raise_for_status()
        return {"ok": True, "latency_s": time.perf_counter() - t0}
    except Exception as e:
        return {"ok": False, "latency_s": time.perf_counter() - t0, "error": str(e)}

def p95(latencies):
    s = sorted(latencies)
    idx = max(0, int(len(s) * 0.95) - 1)
    return s[idx]

async def naive_burst(base_url, model, prompt, n=50, max_tokens=128):
    async with httpx.AsyncClient(timeout=120.0) as client:
        results = await asyncio.gather(*[
            send_one(client, base_url, model, prompt, max_tokens) for _ in range(n)
        ])
    latencies = [r["latency_s"] for r in results if r["ok"]]
    return {
        "n_sent": n,
        "n_ok": len(latencies),
        "p95_s": round(p95(latencies), 3) if latencies else None,
        "mean_s": round(sum(latencies) / len(latencies), 3) if latencies else None,
    }

prompt = "In two sentences, explain what a load balancer does."
naive_result = await naive_burst(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt=prompt, n=50,
)
print("naive (unbounded):", naive_result)

naive (unbounded): {'n_sent': 50, 'n_ok': 50, 'p95_s': 0.933, 'mean_s': 0.923}


In [8]:
class LoadShedder:
    def __init__(self, max_in_flight: int):
        self.sem = asyncio.Semaphore(max_in_flight)

    async def try_admit(self):
        # non-blocking acquire: True if a slot was free right now, False if not
        acquired = self.sem.locked() is False and self.sem._value > 0
        if acquired:
            await self.sem.acquire()
        return acquired

    def release(self):
        self.sem.release()

async def send_with_shedding(client, shedder, base_url, model, prompt, max_tokens=128):
    admitted = await shedder.try_admit()
    if not admitted:
        return {"ok": False, "shed": True, "latency_s": 0.0}
    try:
        t0 = time.perf_counter()
        r = await client.post(f"{base_url}/v1/chat/completions".replace("/v1/v1", "/v1"),
                                json={"model": model, "messages": [{"role": "user", "content": prompt}],
                                      "max_tokens": max_tokens, "temperature": 0.0})
        r.raise_for_status()
        return {"ok": True, "shed": False, "latency_s": time.perf_counter() - t0}
    except Exception as e:
        return {"ok": False, "shed": False, "latency_s": time.perf_counter() - t0, "error": str(e)}
    finally:
        shedder.release()

async def shedded_burst(base_url, model, prompt, n=50, cap=8, max_tokens=128):
    shedder = LoadShedder(cap)
    async with httpx.AsyncClient(timeout=120.0) as client:
        results = await asyncio.gather(*[
            send_with_shedding(client, shedder, base_url, model, prompt, max_tokens)
            for _ in range(n)
        ])
    accepted = [r for r in results if r["ok"]]
    shed = [r for r in results if r.get("shed")]
    latencies = [r["latency_s"] for r in accepted]
    return {
        "n_sent": n, "cap": cap,
        "n_accepted": len(accepted), "n_shed": len(shed),
        "accepted_p95_s": round(p95(latencies), 3) if latencies else None,
    }

shedded_result = await shedded_burst(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt=prompt, n=50, cap=8,
)
print("shedded (cap=8):", shedded_result)

shedded (cap=8): {'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.572}


In [9]:
sweep = []
for n in (8, 16, 32, 50):
    r = await shedded_burst(
        base_url="http://localhost:8000/v1",
        model="Qwen/Qwen2.5-1.5B-Instruct",
        prompt=prompt, n=n, cap=8,
    )
    sweep.append(r)
    print(r)

{'n_sent': 8, 'cap': 8, 'n_accepted': 8, 'n_shed': 0, 'accepted_p95_s': 0.654}
{'n_sent': 16, 'cap': 8, 'n_accepted': 8, 'n_shed': 8, 'accepted_p95_s': 0.391}
{'n_sent': 32, 'cap': 8, 'n_accepted': 8, 'n_shed': 24, 'accepted_p95_s': 0.401}
{'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.397}


In [10]:
import json
report = {
    "naive_unbounded_n50": naive_result,
    "shedded_cap8_n50": shedded_result,
    "shedded_sweep": sweep,
}
with open("shedding_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "naive_unbounded_n50": {
    "n_sent": 50,
    "n_ok": 50,
    "p95_s": 0.933,
    "mean_s": 0.923
  },
  "shedded_cap8_n50": {
    "n_sent": 50,
    "cap": 8,
    "n_accepted": 8,
    "n_shed": 42,
    "accepted_p95_s": 0.572
  },
  "shedded_sweep": [
    {
      "n_sent": 8,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 0,
      "accepted_p95_s": 0.654
    },
    {
      "n_sent": 16,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 8,
      "accepted_p95_s": 0.391
    },
    {
      "n_sent": 32,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 24,
      "accepted_p95_s": 0.401
    },
    {
      "n_sent": 50,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 42,
      "accepted_p95_s": 0.397
    }
  ]
}


In [11]:
!python verify.py

invariants hold: shedding happened, accepted p95 protected, cap flat
GREEN CHECK: PASS
